# Setup

This is the rough running file for our actual project of translating from english to german using our native transformer implementation. This will keep getting updated as we make progress.

## Imports

In [41]:
# Reload selfutil and get funcs to use
import importlib
import selfutil

importlib.reload(selfutil)
from selfutil import get_dataset, scaled_dot_product_attention

# Import classes
from classes.LangDataLoader import LangDataLoader
from classes.EmbeddingLayer import EmbeddingLayer
from classes.MultiHeadAttention import MultiHeadAttentionLayer
from classes.PoswiseFeedForward import PoswiseFeedForward

# Other libraries
from datasets import load_dataset
import os
import json
from tqdm import tqdm
import torch
from torch import nn
from transformers import AutoConfig, AutoTokenizer

# Data Setup

## Retrieve the dataset

Firstly we will just load the English-German translation dataset from Hugging Face's datasets library and use a small subset for our training purposes. For this reason we are using the 'de-en' set from the 'wmt14' dataset.

Our function used here takes in the name of the data directory and checks if a file of naming convention dataset-name_config-name_split_num-samples.json exists so if we are using the 'wmt14' dataset, 'de-en' config, 'train' split, 100 samples then it will check specifically if a file called 'wmt14_de-en_train_100.json' exists in our data_dir. If it does then it will use this json for our purposes, if not then it will go on to create the subset in the directory.

In [24]:
# Define parameters for getting the dataset
data_dir = 'data/'
dataset_name = 'wmt14'
config_name = 'de-en'
split = 'train'
num_samples = 100

# Retrieve the dataset and observe length
train_data = get_dataset('data/', 'wmt14', 'de-en', 'train', 100)
print(f'\n1 Sample from {len(train_data)}:\n{train_data[0]}')

JSON EXISTS, loading from data/wmt14_de-en_train_100.json

1 Sample from 100:
{'de': 'Wiederaufnahme der Sitzungsperiode', 'en': 'Resumption of the session'}


## Define Model, Tokenizer params

We will be using the 'bert-base-uncased' model from Hugging Face so we need to define our tokenizer etc. from there

In [3]:
# Define the model name
model_name = 'bert-base-uncased'

# Load the pretrained config and tokenizer
config = AutoConfig.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

/Users/farzanmirza/miniconda3/lib/python3.12/site-packages/huggingface_hub-0.23.4-py3.8.egg/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.


## Convert Dataset into DataLoader Object for use

In [4]:
# Create the DataLoader for training data
train_loader = LangDataLoader(train_data, tokenizer)

# Print the shape of toks
print(f'Shape of English tokens: {train_loader._en_toks.shape}')
print(f'Shape of German tokens: {train_loader._de_toks.shape}')

Tokenizing and Padding: 100%|██████████| 100/100 [00:00<00:00, 2691.50it/s]

Shape of English tokens: torch.Size([100, 256])
Shape of German tokens: torch.Size([100, 256])


# Encoder

## Input Embeddings (English)

Token + Position Embeddings

In [25]:
# Create embedding layer to test
embedding_layer = EmbeddingLayer(config)

# Now convert the English tokens into embeddings and print the shape along with a single example
en_embed = embedding_layer(train_loader._en_toks)
print(f'Shape of English embeddings: {en_embed.shape}\n{en_embed[0]}')

Shape of English embeddings: torch.Size([100, 256, 768])
tensor([[-0.0000, -1.5087, -2.2412,  ...,  0.3066,  0.1939,  0.0000],
        [-2.6637, -1.5018,  0.0000,  ...,  0.4975, -0.0000, -0.0000],
        [-1.3743, -0.0000, -0.0000,  ...,  0.0000,  1.6347,  0.0000],
        ...,
        [ 0.0000,  1.0326,  0.0000,  ..., -0.0000,  0.7740, -0.0000],
        [ 0.5056,  3.5492, -0.0000,  ..., -0.0000, -0.0000, -0.6276],
        [ 0.4008,  0.0000,  2.2189,  ..., -0.0000,  0.2326, -0.0000]],
       grad_fn=<SelectBackward0>)


## Multi-Head Attention 

Run the batch through multi-head attention see what happens

In [32]:
# Create a multi-head attention layer
multihead_attn_layer = MultiHeadAttentionLayer(config)

# Pass a batch of input embeddings through the attention head
multihead_attn = multihead_attn_layer(en_embed)

# Print the output along with its shape
print(f'Single example of Multi-Head Attention sized {multihead_attn.shape}\n{multihead_attn[0]}')

Single example of Multi-Head Attention sized torch.Size([100, 256, 768])
tensor([[-0.2128, -0.1504, -0.0698,  ...,  0.2342,  0.0013, -0.5324],
        [-0.2145, -0.1738, -0.0673,  ...,  0.2036,  0.0127, -0.5229],
        [-0.2153, -0.1410, -0.0734,  ...,  0.2442, -0.0234, -0.5427],
        ...,
        [-0.2015, -0.1229, -0.0760,  ...,  0.2225, -0.0104, -0.5233],
        [-0.2033, -0.1637, -0.0806,  ...,  0.2335, -0.0462, -0.5217],
        [-0.2172, -0.1102, -0.0746,  ...,  0.2079, -0.0182, -0.5413]],
       grad_fn=<SelectBackward0>)


# Position-wise Feed Forward

The Positionwise Feed-Forward Network (FFN) in the Transformer architecture is a crucial component that processes each token's embedding independently across the sequence. Unlike other types of feed-forward networks that may combine information across different dimensions, the Positionwise FFN applies the same linear transformations and non-linear activation to each token individually, without interaction between them. This allows the network to efficiently and independently enrich the representations of each token while preserving the sequence structure. The simplicity of this approach enables parallel processing of tokens, making it computationally efficient and well-suited for the highly parallelizable nature of Transformer models.

In [42]:
# Create a position-wise feed-forward layer
ffn_layer = PoswiseFeedForward(config)

# Pass a batch of input embeddings through the position-wise feed-forward network
ffn_output = ffn_layer(multihead_attn)

# Print the output along with its shape
print(f'Single example of Position-Wise Feed-Forward Network sized {ffn_output.shape}\n{ffn_output[0]}')

Single example of Position-Wise Feed-Forward Network sized torch.Size([100, 256, 768])
tensor([[-0.0462,  0.0591, -0.0448,  ..., -0.0390, -0.0182, -0.0529],
        [-0.0517,  0.0642, -0.0482,  ..., -0.0291, -0.0256, -0.0540],
        [-0.0486,  0.0643, -0.0518,  ..., -0.0331, -0.0176, -0.0597],
        ...,
        [-0.0507,  0.0625, -0.0427,  ..., -0.0396, -0.0207, -0.0538],
        [-0.0469,  0.0567, -0.0494,  ..., -0.0381, -0.0143, -0.0566],
        [-0.0461,  0.0665, -0.0469,  ..., -0.0358, -0.0223, -0.0555]],
       grad_fn=<SelectBackward0>)
